<a href="https://colab.research.google.com/github/zaetae/regime-aware-ml-trading/blob/main/17.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys, os
from pathlib import Path

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

if 'google.colab' in str(getattr(sys, 'modules', {})) or os.path.exists('/content'):
    REPO_DIR  = '/content/regime-aware-ml-trading'
    PROJ_ROOT = os.path.join(REPO_DIR, 'regime-aware-ml-trading')
    if not os.path.isdir(PROJ_ROOT):
        os.system('git clone https://github.com/zaetae/regime-aware-ml-trading.git ' + REPO_DIR)
    else:
        os.system(f'cd {REPO_DIR} && git pull -q')
    os.system(f'{sys.executable} -m pip install -q yfinance hmmlearn scikit-learn seaborn statsmodels')
else:
    def _find_project_root():
        current = Path.cwd()
        for _ in range(10):
            if (current / "src").is_dir():
                return current
            current = current.parent
        return Path.cwd().parent if (Path.cwd().parent / "src").is_dir() else Path.cwd()
    PROJ_ROOT = str(_find_project_root())

sys.path.insert(0, PROJ_ROOT)
os.chdir(PROJ_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Project root:", PROJ_ROOT)
print("src exists:", os.path.isdir(os.path.join(PROJ_ROOT, 'src')))

# 17 — Consolidated Multi-Asset Detector & Labeling Summary

Final validation notebook using the fully corrected detector pipeline:
- Channels: symmetric 2/2 touch requirement, 0.15×ATR tolerance
- Triangles: window=20, pivot_order=2, relative flat_threshold_mult=0.25
- S/R: unchanged (investigated, confirmed genuine market behavior, not a bug)

Run across 10 instruments (SPY + 9 pooled tickers) to produce final,
authoritative event counts, label distributions, and training data for
the multi-asset generalization tests reported in notebook 14.

In [ ]:
import inspect
from src.patterns.channels import detect_channel
from src.patterns.triangles import detect_triangle_pattern

print("Channel defaults:", inspect.signature(detect_channel))
print("Triangle defaults:", inspect.signature(detect_triangle_pattern))

In [ ]:
import yfinance as yf
import pandas as pd
from src.patterns.scanner import scan_all_patterns
from src.labeling.label_events import label_events
from src.features.build_features import build_feature_matrix

def process_ticker(ticker, start='2010-01-01', end='2026-01-01'):
    raw = yf.download(ticker, start=start, end=end, auto_adjust=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.droplevel(1)
    raw = raw[['Open', 'High', 'Low', 'Close', 'Volume']]
    if raw.index.tz is not None:
        raw.index = raw.index.tz_localize(None)
    raw.index.name = 'Date'

    scanned = scan_all_patterns(raw.copy())
    features, labels, labeled_df = build_feature_matrix(raw.copy())

    return {
        'ticker': ticker, 'raw': raw, 'scanned': scanned,
        'features': features, 'labels': labels, 'labeled_df': labeled_df,
    }

TICKERS = ['SPY', 'QQQ', 'NVDA', 'GOOGL', 'AAPL', 'MSFT', 'AMZN', 'META', 'JPM', 'XOM']

results_final = {}
for t in TICKERS:
    try:
        results_final[t] = process_ticker(t)
        print(f"{t}: OK — {results_final[t]['scanned']['has_event'].sum()} events, "
              f"{results_final[t]['features'].shape[0]} labeled")
    except Exception as e:
        print(f"{t}: FAILED — {e}")

In [ ]:
summary_final = pd.DataFrame([
    {
        'Ticker': t,
        'Bars': len(r['raw']),
        'Near Support': int(r['scanned']['near_support'].sum()),
        'Near Resistance': int(r['scanned']['near_resistance'].sum()),
        'Triangles': int(r['scanned']['triangle_pattern'].notna().sum()),
        'Channels': int(r['scanned']['channel_pattern'].notna().sum()),
        'Multi Top/Bottom': int(r['scanned']['multiple_top_bottom_pattern'].notna().sum()),
        'Total Events': int(r['scanned']['has_event'].sum()),
        'Labeled Events': r['features'].shape[0],
        'Long %': round((r['labels'] == 'long').mean() * 100, 1),
        'Short %': round((r['labels'] == 'short').mean() * 100, 1),
        'No-trade %': round((r['labels'] == 'no_trade').mean() * 100, 1),
    }
    for t, r in results_final.items()
])
print(summary_final.to_string(index=False))